# Отчет о проделанной работе
# Распознавание рукописной математики



### Цель:
получение обученной модели-преобразователя, способной переводить изображение рукописной формулы в корректную строку LaTeX.


### Задачи:
1)	Произвести дообучение любой VL модели для конвертации изображения с рукописной формулой в формат LaTeX.
2)	Разработать Streamlit-приложение с функционалом drag-and-drop.
Практическая часть

## Практическая часть

### Базовая модель:
Qwen2.5-VL-3B-Instinct

### Подход:
LoRA fine-tunung. Было произведено дообучение ~7 млн. параметров из 3 млрд (0.196 %). на датасетах с изображениями и лэйблами.
Главная идея LoRA - не изменять все веса модели, а обучать только малые дополнительные матрицы-надстройки поверх некоторых слоёв, в интересах смещения её поведения в сторону конкретной задачи.

### Блок-схема проекта:
#### Блок кодготовки данных
Взятие датасета и приведение к единому формату.
#### Блок обучения
Поверх базовой модели лежит LoRA-адаптер, который обучается
#### Инференс
Этап запуска. Здесь производится следующее:
- Загрузка датасета;
- формирование проекта;
- прогон через модель;
- получение предсказанного LaTeX.
#### Интерфейс и демонстрация
Доведение решения до пользовательского сфенария. Streamlit-приложения имеет следующие возможности:
- загрузка изображения;
- выгрузка предсказанной формулы.

## Работа инференса по шагам:
1. Загрузка базовой мультимодальной модели.
2. Поверх неё подключается LoRA-адаптер.
3. Берётся изображение формулы.
4. Формируется промт: "Convert the handwritten mathematical formula into valid LaTeX."
5. Изображение + текстовый промт подаются в модель.
6. Модель генерирует ответ.
7. Ответ декодируется как строка.
8. Происходит отображение строки в друх видах: LaTeX-текстом и конвертированной формулой.

## Проверка результата
Сначала было протестировано использование дообученной модели на одном изображении в пользовательском сценарии при использовании Streamlit-приложения. Результаты предложены ниже:

![](./Examples/Result1.png)

На этом же примере была протестирована модель без LoRA-адаптера. Результаты сравнения двух ответов приведены ниже:

![](./Examples/Result1_comparing.png)



После небольшого тестирования было произведенено оценивание результата модели следующими способами:
- zero-shot - модель без SFT. Предсказывает ответ без единого примера;
- one-shot - модель тоже без SFT. Прежде, чем приступать к предсказанию, модели дают один эталон;
- SFT using linxy/LaTeX_OCR:train + deepcopy/MathWriting-human - модель дообучена на датасетах "linxy/LaTeX_OCR:train" и "deepcopy/MathWriting-human".

По условию задания требовалось дообучить модель на одном датасете linxy/LaTeX_OCR:train, но это оказалось невозможным из-за технических ограничений.

Скрипт evaluate.py по окончании оценивания возвращает две метрики:
- Exact Match - показывает, в каком проценте случаев предсказание модели полностью совпало с правильным ответом.
- Normalized Edit Distance - показывает, насколько предсказанный текст близок к правильному, даже если полного совпадения нет.

### Итоги:

#### Zero-shot:
- Exact Match = 0.0000
- Normalized Edit Distance = 0.2257

#### One-shot:
- Exact Match = 0.0857
- Normalized Edit Distance = 0.2084

#### SFT using linxy/LaTeX_OCR:train + deepcopy/MathWriting-human:
- Exact Match = 0.1714
- Normalized Edit Distance = 0.4510



## Выводы:
В результате проделанной работы был разработан обученный LoRA-адаптер.

Сильные стороны:
- проект прикладной и понятный;
- существуют способы применения и развития;
- есть сравнение базовой модели с дообученной;
- есть демонстрация пользовательского сценария через Streamlit;
- есть end-to-end пайплайнЖ от данных  до интерфейса;

Ограничения:
- качество зависит от почерка;
- LoRA зависит от базовой модели;
- В некоторых скриптах имеют захардкодерные места, например, путь до директории.

P.S. Последний пункт ограничения будет доработан в ближайшем будущем.